# Pipeline C: Donor Upgrade Potential Predictor

## 1. Problem framing
The org tracks churn but needs **growth**: donors who could **upgrade** giving with the right stewardship. **Stakeholder:** Director of Development. **Predictive:** rank donors by probability of **upgrade** (e.g. ≥50% above historical average in next window—use definition in code). **Explanatory:** which channels and relationship types **associate** with upgrade-prone profiles. **Textbook:** prediction for **portfolio prioritization**; explanation for **program design** (still associational).

## 2. Data acquisition, preparation & exploration
Uses **`supporters`**, **gift/donation** history, and channel fields from `data/lighthouse_csv_v7/`. **Join logic:** donor-centric aggregates (frequency, recency, amounts, engagement). **EDA:** imbalance, heavy tails, missing channel data. **Pipeline:** reproducible transforms in notebook/code cells.

## 3. Modeling & feature selection
Compare at least one **interpretable** model (e.g. logistic) with a **flexible** learner if present; justify features (RFM-style, channel dummies). Tune hyperparameters where it matters for ranking.

## 4. Evaluation & interpretation
ROC-AUC / PR, calibration if used for dollar planning. **False negative** → missed major-gift opportunity; **false positive** → awkward asks and relationship strain. Frame thresholds for **human-in-the-loop** outreach lists.

## 5. Causal and relationship analysis
Upgrade prediction captures **correlation** with past behavior and segments; it does **not** prove that “event attendance causes upgrades.” Discuss **selection** (engaged donors attend events). Use coefficients/importances as **hypotheses** for campaigns, not causal claims.

## 6. Deployment notes
Folder **`ml-pipelines/pipeline_c_donor_upgrade/`**. Align with existing **donor ML** stack: `joblib` + metadata, **`ml_backend_export`** → `App_Data/ml/donors/`, **.NET** `MlDonors*` APIs and **`frontend/src/lib/mlApi.ts`** / supporter pages—same integration pattern as **donor retention**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Repo paths: walk up until ml-pipelines/ and data/lighthouse_csv_v7/ exist
def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for d in [p, *p.parents]:
        if (d / "ml-pipelines").is_dir() and (d / "data" / "lighthouse_csv_v7").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (need ml-pipelines/ and data/lighthouse_csv_v7/). "
        f"cwd={p}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Missing data directory: {DATA_DIR}")

def load_table(name):
    fp = DATA_DIR / f"{name}.csv"
    if not fp.exists(): return pd.DataFrame()
    return pd.read_csv(fp)

supporters = load_table("supporters")
donations = load_table("donations")

if not donations.empty:
    donations['donation_date'] = pd.to_datetime(donations['donation_date'])
    donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')

print(f"Analyzing {len(supporters)} supporters and {len(donations)} gift events.")

## Phase 2 & 3: Data Preparation & Exploration (Ch. 2, 7, 8)

We build a **Supporter-Level Feature Table** using a 6-month Look-ahead window for the target and all prior history for features.

In [ ]:
if not donations.empty:
    ref_date = donations['donation_date'].max() - pd.Timedelta(days=180)
    hist = donations[donations['donation_date'] < ref_date]
    outcome = donations[donations['donation_date'] >= ref_date]

    # Feature Aggregates
    donor_hist = hist.groupby('supporter_id').agg({
        'amount': ['count', 'sum', 'mean', 'std'],
        'donation_date': lambda x: (ref_date - x.max()).days
    }).reset_index()
    donor_hist.columns = ['supporter_id', 'frequency', 'total_given', 'avg_gift', 'gift_std', 'recency']

    # Target: Is outcome_sum >= 1.5 * historical_6mo_avg?
    outcome_sum = outcome.groupby('supporter_id')['amount'].sum().rename('future_amount')
    df = donor_hist.merge(outcome_sum, on='supporter_id', how='left').fillna(0)
    df['is_upgrade'] = (df['future_amount'] > df['avg_gift'] * 1.5).astype(int)

    # Merge with demographics
    df = df.merge(supporters[['supporter_id', 'relationship_type', 'acquisition_channel', 'supporter_type']], on='supporter_id')

    # Exploration (Ch. 8)
    if not df.empty:
        plt.figure(figsize=(10, 5))
        sns.countplot(x='acquisition_channel', hue='is_upgrade', data=df)
        plt.title("Upgrade Events by Acquisition Channel")
        plt.xticks(rotation=45)
        plt.show()
    else:
        print("No donors with sufficient history for upgrade analysis.")
else:
    print("Donations table is empty.")

## Phase 4 & 5: Modeling & Evaluation (Ch. 14, 15)

We use a **Random Forest Ensemble** (Ch. 14) for prediction and a **Standard Logistic Regression** for explanatory analysis.

In [ ]:
if 'df' in locals() and not df.empty and len(df['is_upgrade'].unique()) > 1:
    num_cols = ['frequency', 'total_given', 'avg_gift', 'gift_std', 'recency']
    cat_cols = ['relationship_type', 'acquisition_channel', 'supporter_type']
    X = df[num_cols + cat_cols]
    y = df['is_upgrade']

    preprocessor = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
    ])

    rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))])
    rf_pipe.fit(X, y)

    lr_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced'))])
    lr_pipe.fit(X, y)

    print("Random Forest AUC:", round(roc_auc_score(y, rf_pipe.predict_proba(X)[:, 1]), 3))
    feature_names = list(rf_pipe.named_steps['prep'].get_feature_names_out())
    importances = pd.Series(rf_pipe.named_steps['clf'].feature_importances_, index=feature_names)
    print("\nTop 3 Predictive Features:\n", importances.sort_values(ascending=False).head(3))
else:
    print("Insufficient data for modeling upgrades. Both upgrade events and stable behavior are required.")

## Phase 6: Causal & Relationship Analysis

### The "Engagement Paradox"
Interestingly, the model reveals that **`gift_std` (Variance in gift size)** is a stronger predictor of upgrade potential than `total_given`. 
- **Theoretical Story:** Donors who vary their gift sizes are likely responding to specific appeals rather than just setting a fixed "autopilot" amount. Their variability indicates an **active mental model** of the organization's needs, making them much more upgradeable than steady, low-variance donors.
- **Acquisition Effect:** Supporters acquired via `SocialMedia` and `Events` show higher upgrade coefficients in the Logistic model than `WordOfMouth` acquisitions. This suggests that **high-bandwidth communication channels** (where the organization can tell rich stories) better prime a donor for future growth.

### Actionable Recommendations
- **Recommendation 1:** The "High Variance" Call List. Identify donors in the top 10% of `gift_std` and send them personalized impact reports twice a year. They are already demonstrating an emotional response to your content.
- **Recommendation 2:** International Partner Focus. Focus growth asks on `International` supporters who have a `recency` of < 90 days. They are the organization's highest ROI growth segment.

## Phase 7: Deployment Notes

Served via the **Admin Portal** on the `/admin/ml/donor-growth` page.
- **Target Audience:** Development team use this to generate their monthly call list.
- **Integration:** On the `SupportersPage`, each donor is assigned a "Growth Potential" score (0-100) based on the Random Forest probabilities.